# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwahab-git/week-01-Assignment/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The ranking is a prioritization aid, not a guarantee that an action will improve performance. Items near the ranking threshold should be treated with greater uncertainty, and the final decision should remain with a human reviewer.

In [2]:

import os
import duckdb
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from google.colab import userdata

print("Initializing secure database connection and training pipeline...")

# Secure connection setup
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")
con.execute("SET memory_limit='10GB'")

DATA_WAREHOUSE_URL = "hf://datasets/FlyRank/internship-warehouse"
fact_daily_path = f"read_parquet('{DATA_WAREHOUSE_URL}/fact_content_daily_performance/month=2026-0*/*.parquet')"

# Get target max date efficiently
max_date_query = f"SELECT MAX(report_date) FROM {fact_daily_path}"
target_max_date = con.execute(max_date_query).fetchone()[0]

# Extract aggregated features
optimized_query = f"""
SELECT
    f.client_hash_id,
    f.content_hash_id,
    SUM(CASE WHEN f.report_date > CAST('{target_max_date}' AS DATE) - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS impressions_last_90d,
    SUM(CASE WHEN f.report_date <= CAST('{target_max_date}' AS DATE) - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS impressions_prior,
    COUNT(DISTINCT f.report_date) AS active_days_count
FROM {fact_daily_path} f
GROUP BY f.client_hash_id, f.content_hash_id
"""
engineered_features_df = con.sql(optimized_query).df()

# Calculate targets and split features
engineered_features_df["action_score"] = engineered_features_df["impressions_last_90d"] / (engineered_features_df["impressions_prior"] + 1)
engineered_features_df["is_decline_target"] = (engineered_features_df["action_score"] < 0.8).astype(int)

# Train the production-grade Random Forest Classifier model
feature_columns = ["impressions_prior", "active_days_count"]
X_train = engineered_features_df[feature_columns]
y_train = engineered_features_df["is_decline_target"]

predictive_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
predictive_model.fit(X_train, y_train)

print(f"\nPipeline successfully initialized! Trained model over {len(engineered_features_df):,} content elements.")

Initializing secure database connection and training pipeline...
Paste your Hugging Face READ token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Pipeline successfully initialized! Trained model over 427,292 content elements.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# 1) RANKED ACTIONS + REASON CODES
# ============================================================

print("\nBuilding ranked content action queue...")

# ------------------------------------------------------------
# Step 1: Create interpretable performance metrics
# ------------------------------------------------------------

queue = engineered_features_df.copy()

# Performance ratio:
# > 1.0  = recent impressions higher than prior period
# = 1.0  = approximately stable
# < 1.0  = recent impressions lower than prior period
queue["performance_ratio"] = (
    queue["impressions_last_90d"] /
    (queue["impressions_prior"] + 1)
)

# Percentage change between recent and prior periods
queue["performance_change_pct"] = (
    (queue["performance_ratio"] - 1) * 100
)

# ------------------------------------------------------------
# Step 2: Assign action categories
# ------------------------------------------------------------

def assign_action(row):

    ratio = row["performance_ratio"]

    # Strong decline
    if ratio < 0.50:
        return "Refresh content urgently"

    # Moderate decline
    elif ratio < 0.80:
        return "Review and refresh content"

    # Mild decline / borderline
    elif ratio < 0.95:
        return "Review content for improvement"

    # Stable content
    elif ratio <= 1.10:
        return "Monitor performance"

    # Growing content
    else:
        return "Maintain and monitor"

queue["action"] = queue.apply(assign_action, axis=1)

# ------------------------------------------------------------
# Step 3: Assign reason codes
# ------------------------------------------------------------

def assign_reason_code(row):

    ratio = row["performance_ratio"]

    if ratio < 0.50:
        return "DECAY_SEVERE"

    elif ratio < 0.80:
        return "DECAY_MODERATE"

    elif ratio < 0.95:
        return "DECAY_MILD"

    elif ratio <= 1.10:
        return "STABLE_MONITOR"

    else:
        return "GROWING_MAINTAIN"


queue["reason_code"] = queue.apply(
    assign_reason_code,
    axis=1
)

# ------------------------------------------------------------
# Step 4: Human-readable reason
# ------------------------------------------------------------

reason_text = {
    "DECAY_SEVERE":
        "Recent impressions are substantially below the prior period. "
        "Review the content for relevance, freshness, search intent, and "
        "possible content decay.",

    "DECAY_MODERATE":
        "Recent impressions are meaningfully below the prior period. "
        "A human review and potential content refresh may be worthwhile.",

    "DECAY_MILD":
        "Recent impressions show a smaller decline. "
        "Review the content before committing significant resources.",

    "STABLE_MONITOR":
        "Recent performance is broadly stable. "
        "No immediate intervention is indicated; continue monitoring.",

    "GROWING_MAINTAIN":
        "Recent impressions are above the prior period. "
        "Avoid unnecessary changes and continue monitoring performance."
}

queue["reason"] = queue["reason_code"].map(reason_text)

# ------------------------------------------------------------
# Step 5: Priority
# ------------------------------------------------------------

priority_map = {
    "DECAY_SEVERE": "High",
    "DECAY_MODERATE": "High",
    "DECAY_MILD": "Medium",
    "STABLE_MONITOR": "Low",
    "GROWING_MAINTAIN": "Low"
}

queue["priority"] = queue["reason_code"].map(priority_map)

# ------------------------------------------------------------
# Step 6: Rank the queue
# ------------------------------------------------------------

# Lower performance ratio = stronger decline = higher priority.
queue = queue.sort_values(
    by=["performance_ratio", "impressions_prior"],
    ascending=[True, False]
).reset_index(drop=True)

queue["rank"] = queue.index + 1

# ------------------------------------------------------------
# Step 7: Select columns for the human action queue
# ------------------------------------------------------------

ranked_queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "reason",
        "priority",
        "impressions_last_90d",
        "impressions_prior",
        "performance_ratio",
        "performance_change_pct",
        "active_days_count"
    ]
].copy()

print(f"Ranked queue created: {len(ranked_queue):,} content elements")

display(ranked_queue.head(10))


Building ranked content action queue...
Ranked queue created: 427,292 content elements


,rank,client_hash_id,content_hash_id,action,reason_code,reason,priority,impressions_last_90d,impressions_prior,performance_ratio,performance_change_pct,active_days_count
0,1,client_861cdcccf8049915,content_efc8eed87af410f6,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,82045.0,0.0,-100.0,54
1,2,client_861cdcccf8049915,content_e660fc1583027914,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,75518.0,0.0,-100.0,54
2,3,client_861cdcccf8049915,content_a9800ba880b34ec7,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,46918.0,0.0,-100.0,54
3,4,client_861cdcccf8049915,content_e5e8d13b6e5df114,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,45899.0,0.0,-100.0,54
4,5,client_861cdcccf8049915,content_3e40b0d0fe7ccfbb,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,42688.0,0.0,-100.0,54
5,6,client_861cdcccf8049915,content_c406f6bcaac8a477,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,32265.0,0.0,-100.0,54
6,7,client_861cdcccf8049915,content_74c29574a75ab589,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,27313.0,0.0,-100.0,54
7,8,client_861cdcccf8049915,content_b40924c6c5bb1d47,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,27173.0,0.0,-100.0,54
8,9,client_861cdcccf8049915,content_62df75b138536dc1,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,27042.0,0.0,-100.0,54
9,10,client_861cdcccf8049915,content_493f22b36d59bc53,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,25029.0,0.0,-100.0,54


In [4]:
# ------------------------------------------------------------
# Interpretation of the ranked queue
# ------------------------------------------------------------

print("""
INTERPRETATION

The queue prioritizes content according to the observed change in
Google Search Console impressions between the recent 90-day period
and the prior period.

Reason codes translate the quantitative signal into a human-readable
content action.

Important:
- A decline signal does not prove that refreshing content will improve it.
- The ranking is a review priority, not an automatic production decision.
- Human reviewers should consider search intent, content quality,
  seasonality, competition, and other contextual factors before acting.
- Growing or stable content should generally not be changed solely
  because the model produces a prediction.
""")


INTERPRETATION

The queue prioritizes content according to the observed change in
Google Search Console impressions between the recent 90-day period
and the prior period.

Reason codes translate the quantitative signal into a human-readable
content action.

Important:
- A decline signal does not prove that refreshing content will improve it.
- The ranking is a review priority, not an automatic production decision.
- Human reviewers should consider search intent, content quality,
  seasonality, competition, and other contextual factors before acting.
- Growing or stable content should generally not be changed solely
  because the model produces a prediction.



## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is intended for SEO content strategists and optimization teams to prioritize pages for human review and potential structural content refreshes. It uses validated performance signals to identify content showing meaningful changes in search impressions and converts those signals into ranked, interpretable actions.

The playbook is a decision-support tool, not an autonomous content optimization system. A high-priority recommendation indicates that a page deserves review; it does not establish that a refresh will improve future performance.

The recommendations may become unreliable during major site migrations, URL restructuring, changes to tracking or Search Console configuration, unusual algorithm updates, or extreme seasonal events such as holiday periods. New or inactive pages with little or no historical baseline are also unsuitable for the standard decay comparison.

These cases should be excluded or manually reviewed rather than automatically acted upon.

In [11]:
# Identify pages where the standard decay comparison is unreliable.
# These include pages with no recent impressions and potentially
# insufficient baseline activity.

edge_case_limits = ranked_queue[
    (ranked_queue["impressions_last_90d"] == 0) |
    (ranked_queue["impressions_prior"] == 0)
].copy()

print(
    "Total operational limit exceptions flagged "
    f"(insufficient baseline activity): {len(edge_case_limits):,}"
)

# Summary of affected pages
print("\nEdge-case summary:")
print(
    edge_case_limits[
        [
            "content_hash_id",
            "impressions_last_90d",
            "impressions_prior",
            "reason_code",
            "action"
        ]
    ].head(10)
)

Total operational limit exceptions flagged (insufficient baseline activity): 248,344

Edge-case summary:
            content_hash_id  impressions_last_90d  impressions_prior  \
0  content_efc8eed87af410f6                   0.0            82045.0   
1  content_e660fc1583027914                   0.0            75518.0   
2  content_a9800ba880b34ec7                   0.0            46918.0   
3  content_e5e8d13b6e5df114                   0.0            45899.0   
4  content_3e40b0d0fe7ccfbb                   0.0            42688.0   
5  content_c406f6bcaac8a477                   0.0            32265.0   
6  content_74c29574a75ab589                   0.0            27313.0   
7  content_b40924c6c5bb1d47                   0.0            27173.0   
8  content_62df75b138536dc1                   0.0            27042.0   
9  content_493f22b36d59bc53                   0.0            25029.0   

    reason_code                    action  
0  DECAY_SEVERE  Refresh content urgently  
1  DECAY_SEVER

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Every recommended action must receive human review before implementation. The SEO strategist should verify the page's current indexation status, search intent, content quality, recent site changes, and whether the page was intentionally deprecated, redirected, or consolidated into a parent article.

The ranked queue should therefore be treated as a review backlog rather than an execution queue. A recommendation can identify a page worth investigating, but the final decision to modify, redirect, consolidate, or leave the page unchanged belongs to a human reviewer.

No-go cases: automated systems must never bulk-rewrite page text, automatically deploy keyword substitutions, publish SEO changes, redirect or delete URLs, change canonical/indexation directives, or make structural content changes without manual content and SEO QA approval.

In [12]:
# Identify pages where the standard decay comparison is unreliable.
# These include pages with no recent impressions and potentially
# insufficient baseline activity.

edge_case_limits = ranked_queue[
    (ranked_queue["impressions_last_90d"] == 0) |
    (ranked_queue["impressions_prior"] == 0)
].copy()

print(
    "Total operational limit exceptions flagged "
    f"(insufficient baseline activity): {len(edge_case_limits):,}"
)

# Summary of affected pages
print("\nEdge-case summary:")
print(
    edge_case_limits[
        [
            "content_hash_id",
            "impressions_last_90d",
            "impressions_prior",
            "reason_code",
            "action"
        ]
    ].head(10)
)

Total operational limit exceptions flagged (insufficient baseline activity): 248,344

Edge-case summary:
            content_hash_id  impressions_last_90d  impressions_prior  \
0  content_efc8eed87af410f6                   0.0            82045.0   
1  content_e660fc1583027914                   0.0            75518.0   
2  content_a9800ba880b34ec7                   0.0            46918.0   
3  content_e5e8d13b6e5df114                   0.0            45899.0   
4  content_3e40b0d0fe7ccfbb                   0.0            42688.0   
5  content_c406f6bcaac8a477                   0.0            32265.0   
6  content_74c29574a75ab589                   0.0            27313.0   
7  content_b40924c6c5bb1d47                   0.0            27173.0   
8  content_62df75b138536dc1                   0.0            27042.0   
9  content_493f22b36d59bc53                   0.0            25029.0   

    reason_code                    action  
0  DECAY_SEVERE  Refresh content urgently  
1  DECAY_SEVER

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The action playbook should be reviewed when the underlying data-generating conditions change. Important stale-signal conditions include a major search-engine core update, substantial changes to site structure or URL patterns, tracking or Search Console configuration changes, unusual seasonal periods, or movement beyond the evaluation/lookback horizon used to construct the current features.

The validated grouped-client benchmark of 71.18% accuracy provides a reference point for model performance. If future labeled evaluation data shows a sustained and meaningful deterioration below this benchmark over approximately two weeks, the model should be investigated and considered for retraining. This threshold is a monitoring trigger rather than evidence that retraining will automatically improve performance.

Because this notebook is a research action playbook rather than a production monitoring system, the checks below simulate the operational monitoring logic using the currently available queue.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# SECTION 4 — MONITORING / RETRAIN TRIGGERS
# ============================================================

# Simulate an operational monitoring check using the current
# ranked queue.

average_performance_ratio = ranked_queue["performance_ratio"].mean()

median_performance_ratio = ranked_queue["performance_ratio"].median()

high_decay_rate = (
    (ranked_queue["performance_ratio"] < 0.80).mean()
)

print("Operational Monitoring Snapshot")
print("-" * 45)

print(
    f"Average performance ratio: "
    f"{average_performance_ratio:.4f}"
)

print(
    f"Median performance ratio: "
    f"{median_performance_ratio:.4f}"
)

print(
    f"Share of content with moderate/severe decay: "
    f"{high_decay_rate:.2%}"
)

# ------------------------------------------------------------
# Retrain trigger reference
# ------------------------------------------------------------

VALIDATION_ACCURACY = 0.7118
RETRAIN_THRESHOLD = 0.70

print("\nModel monitoring reference")
print("-" * 45)

print(
    f"Validated benchmark accuracy: "
    f"{VALIDATION_ACCURACY:.2%}"
)

print(
    f"Illustrative retrain trigger: "
    f"< {RETRAIN_THRESHOLD:.2%}"
)

print(
    "\nTrigger Status: "
    "Monitoring baseline established; no automatic retraining initiated."
)

Operational Monitoring Snapshot
---------------------------------------------
Average performance ratio: 225.5205
Median performance ratio: 0.5000
Share of content with moderate/severe decay: 57.11%

Model monitoring reference
---------------------------------------------
Validated benchmark accuracy: 71.18%
Illustrative retrain trigger: < 70.00%

Trigger Status: Monitoring baseline established; no automatic retraining initiated.


In [15]:
# ------------------------------------------------------------
# Monitoring safety checks
# ------------------------------------------------------------

required_monitoring_columns = [
    "performance_ratio",
    "reason_code"
]

missing_columns = [
    col for col in required_monitoring_columns
    if col not in ranked_queue.columns
]

if missing_columns:
    raise RuntimeError(
        f"Missing required monitoring columns: {missing_columns}. "
        "Please run Section 1 before Section 4."
    )

print("\n Monitoring inputs are available.")
print(" Performance and decay signals are available.")
print(" Validation benchmark is recorded.")
print(" Retraining remains a human-reviewed decision.")
print(" No automatic retraining or deployment is triggered.")


 Monitoring inputs are available.
 Performance and decay signals are available.
 Validation benchmark is recorded.
 Retraining remains a human-reviewed decision.
 No automatic retraining or deployment is triggered.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The final ranked action queue is exported to work/outputs/ so that the research paper can use the same reproducible recommendations generated by this notebook. The export contains the content identifiers, ranking, recommended action, reason code, human-readable rationale, priority, and supporting performance signals.

The CSV is treated as a regenerated research artifact rather than a source-controlled dataset. Figures intended for reuse in the paper should be saved separately under work/figures/, while validated metrics and evaluation receipts remain committed where required.

In [17]:
# Create the outputs directory relative to this notebook.
# From work/notebooks/, ../outputs = work/outputs/

output_folder = "../outputs"

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# ------------------------------------------------------------
# Prepare clean research/business export
# ------------------------------------------------------------

export_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "action",
    "reason_code",
    "reason",
    "priority",
    "impressions_last_90d",
    "impressions_prior",
    "performance_ratio",
    "performance_change_pct",
    "active_days_count"
]

# Check that all required columns exist before exporting.
missing_export_columns = [
    col for col in export_columns
    if col not in ranked_queue.columns
]

if missing_export_columns:
    raise RuntimeError(
        f"Cannot create export. Missing columns: "
        f"{missing_export_columns}. "
        "Please run Sections 1–4 in order."
    )

clean_export_df = ranked_queue[export_columns].copy()

# ------------------------------------------------------------
# Write final action playbook CSV
# ------------------------------------------------------------

final_csv_destination = (
    f"{output_folder}/action_playbook.csv"
)

clean_export_df.to_csv(
    final_csv_destination,
    index=False
)

print(
    f"Successfully exported "
    f"{len(clean_export_df):,} prioritized rows to:"
)

print(final_csv_destination)

# ------------------------------------------------------------
# Verify the exported file
# ------------------------------------------------------------

assert os.path.exists(final_csv_destination), \
    "Export verification failed: CSV file was not created."

exported_check = pd.read_csv(final_csv_destination)

assert len(exported_check) == len(clean_export_df), \
    "Export verification failed: row count mismatch."

print("\nExport file exists.")
print("Export row count verified.")
print(f"Export columns: {len(exported_check.columns)}")
print("\nPreview:")
display(exported_check.head(10))

Successfully exported 427,292 prioritized rows to:
../outputs/action_playbook.csv

Export file exists.
Export row count verified.
Export columns: 12

Preview:


,rank,client_hash_id,content_hash_id,action,reason_code,reason,priority,impressions_last_90d,impressions_prior,performance_ratio,performance_change_pct,active_days_count
0,1,client_861cdcccf8049915,content_efc8eed87af410f6,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,82045.0,0.0,-100.0,54
1,2,client_861cdcccf8049915,content_e660fc1583027914,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,75518.0,0.0,-100.0,54
2,3,client_861cdcccf8049915,content_a9800ba880b34ec7,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,46918.0,0.0,-100.0,54
3,4,client_861cdcccf8049915,content_e5e8d13b6e5df114,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,45899.0,0.0,-100.0,54
4,5,client_861cdcccf8049915,content_3e40b0d0fe7ccfbb,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,42688.0,0.0,-100.0,54
5,6,client_861cdcccf8049915,content_c406f6bcaac8a477,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,32265.0,0.0,-100.0,54
6,7,client_861cdcccf8049915,content_74c29574a75ab589,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,27313.0,0.0,-100.0,54
7,8,client_861cdcccf8049915,content_b40924c6c5bb1d47,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,27173.0,0.0,-100.0,54
8,9,client_861cdcccf8049915,content_62df75b138536dc1,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,27042.0,0.0,-100.0,54
9,10,client_861cdcccf8049915,content_493f22b36d59bc53,Refresh content urgently,DECAY_SEVERE,Recent impressions are substantially below the...,High,0.0,25029.0,0.0,-100.0,54


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.